In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model="claude-haiku-4-5"


In [2]:
# Test cases for intent classification. Each entry pairs a realistic user
# message with the intent we expect the classifier to pick.
INTENTS = ["refund_request", "order_status", "billing_issue"]

test_cases = [
    {"message": "I want to return my order and get my money back.", "expected_intent": "refund_request"},
    {"message": "Can I get a refund for order #48213? It arrived broken.", "expected_intent": "refund_request"},
    {"message": "I'd like to send this back and be reimbursed.", "expected_intent": "refund_request"},
    {"message": "Where is my package? It hasn't arrived yet.", "expected_intent": "order_status"},
    {"message": "Can you tell me the status of order #77120?", "expected_intent": "order_status"},
    {"message": "Has my order shipped yet?", "expected_intent": "order_status"},
    {"message": "I was charged twice for the same order.", "expected_intent": "billing_issue"},
    {"message": "The amount on my credit card statement doesn't match my receipt.", "expected_intent": "billing_issue"},
    {"message": "Why was I billed $20 more than the price I saw at checkout?", "expected_intent": "billing_issue"},
]

In [3]:
CLASSIFIER_SYSTEM_PROMPT = f"""You classify a customer's message into exactly one intent.
Valid intents: {", ".join(INTENTS)}.
Respond with only the intent label and nothing else."""


def classify_intent(message: str) -> str:
    """Ask Claude to label a single user message with one of INTENTS, deterministically."""
    response = client.messages.create(
        model=model,
        max_tokens=20,
        temperature=0,
        system=CLASSIFIER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": message}],
    )

    text = next(block.text for block in response.content if block.type == "text")
    return text.strip()

In [4]:
def run_intent_eval(cases: list) -> None:
    """Run classify_intent over each test case and report pass/fail plus accuracy."""
    passed = 0

    for case in cases:
        predicted = classify_intent(case["message"])
        is_correct = predicted == case["expected_intent"]
        passed += is_correct

        status = "PASS" if is_correct else "FAIL"
        print(f"[{status}] \"{case['message']}\" -> predicted={predicted!r}, expected={case['expected_intent']!r}")

    print(f"\n{passed}/{len(cases)} passed ({passed / len(cases):.0%})")


run_intent_eval(test_cases)

[PASS] "I want to return my order and get my money back." -> predicted='refund_request', expected='refund_request'
[PASS] "Can I get a refund for order #48213? It arrived broken." -> predicted='refund_request', expected='refund_request'
[PASS] "I'd like to send this back and be reimbursed." -> predicted='refund_request', expected='refund_request'
[PASS] "Where is my package? It hasn't arrived yet." -> predicted='order_status', expected='order_status'
[PASS] "Can you tell me the status of order #77120?" -> predicted='order_status', expected='order_status'
[PASS] "Has my order shipped yet?" -> predicted='order_status', expected='order_status'
[PASS] "I was charged twice for the same order." -> predicted='billing_issue', expected='billing_issue'
[PASS] "The amount on my credit card statement doesn't match my receipt." -> predicted='billing_issue', expected='billing_issue'
[PASS] "Why was I billed $20 more than the price I saw at checkout?" -> predicted='billing_issue', expected='billing_i

In [5]:
# Model-based grader demo. Unlike the intent classifier above (graded by exact
# string match), this grades free-form assistant replies against a rubric —
# useful when there's no single correct output to compare against.

In [6]:
GRADER_SYSTEM_PROMPT = """You are grading a customer support assistant's reply for an online shop.

Judge the reply against this rubric:
1. If the customer wants a return/refund and hasn't given an order number yet, the reply must ask for the order number before proceeding.
2. If the customer already gave an order number, the reply must confirm the return and explain next steps rather than asking for it again.
3. The tone must be polite and professional.

Respond with exactly two lines:
VERDICT: PASS or FAIL
REASON: one sentence explaining why."""


def grade_response(user_message: str, assistant_reply: str) -> dict:
    """Ask Claude to judge a candidate assistant reply against the rubric, deterministically."""
    grading_input = f"Customer message: {user_message}\n\nAssistant reply: {assistant_reply}"
    response = client.messages.create(
        model=model,
        max_tokens=100,
        temperature=0,
        system=GRADER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": grading_input}],
    )

    text = next(block.text for block in response.content if block.type == "text").strip()
    verdict_line, *rest = text.splitlines()
    verdict = "PASS" if "PASS" in verdict_line else "FAIL"
    reason = " ".join(rest).replace("REASON:", "").strip()
    return {"verdict": verdict, "reason": reason}

In [7]:
# Candidate assistant replies to grade, each paired with the verdict a rubric-following grader should reach.
grader_test_cases = [
    {
        "user_message": "I want to return my order and get my money back.",
        "assistant_reply": "I'd be happy to help with that! Could you share your order number so I can start the return?",
        "expected_verdict": "PASS",
    },
    {
        "user_message": "I want to return my order and get my money back.",
        "assistant_reply": "Sure, your refund has been processed.",
        "expected_verdict": "FAIL",  # never asked for the order number
    },
    {
        "user_message": "My order number is #48213, I'd like to return it.",
        "assistant_reply": "Thanks! I've confirmed the return for order #48213. You'll receive a prepaid shipping label by email within 24 hours.",
        "expected_verdict": "PASS",
    },
    {
        "user_message": "My order number is #48213, I'd like to return it.",
        "assistant_reply": "What's your order number?",
        "expected_verdict": "FAIL",  # order number was already given
    },
    {
        "user_message": "I want to return my order.",
        "assistant_reply": "That's not my problem, figure it out yourself.",
        "expected_verdict": "FAIL",  # unprofessional tone
    },
]

In [8]:
def run_model_graded_eval(cases: list) -> None:
    """Grade each candidate reply with the model-based grader and report pass/fail plus accuracy."""
    passed = 0

    for case in cases:
        result = grade_response(case["user_message"], case["assistant_reply"])
        is_correct = result["verdict"] == case["expected_verdict"]
        passed += is_correct

        status = "PASS" if is_correct else "FAIL"
        print(f"[{status}] reply={case['assistant_reply']!r}")
        print(f"    grader_verdict={result['verdict']!r} expected={case['expected_verdict']!r} reason={result['reason']!r}")

    print(f"\n{passed}/{len(cases)} correct ({passed / len(cases):.0%})")


run_model_graded_eval(grader_test_cases)

[PASS] reply="I'd be happy to help with that! Could you share your order number so I can start the return?"
    grader_verdict='PASS' expected='PASS' reason='The reply appropriately asks for the order number before proceeding, maintains a polite and professional tone, and follows the rubric requirement for customers requesting returns without providing an order number.'
[PASS] reply='Sure, your refund has been processed.'
    grader_verdict='FAIL' expected='FAIL' reason='The reply fails to ask for the order number before proceeding, which is required by rubric rule 1 when a customer requests a return/refund without providing an order number.'
[PASS] reply="Thanks! I've confirmed the return for order #48213. You'll receive a prepaid shipping label by email within 24 hours."
    grader_verdict='PASS' expected='PASS' reason='The reply correctly confirms the return with the provided order number and explains the next steps without asking for the order number again, while maintaining a poli